# Use XGBoost to separate morphological features between WT and V53I CCM2 in teloHAECs
# Examine CCM2 subcellular localization


1. **Experimental setup** 
- **Genotypes**: CCM2 WT and coding variant V53I expressed in CCM2KO teloHAECs
- **Conditions**: STATIC / 2HR of shear stress / 24HR of shear stress
- **Biological replicates**: 
  - WT: W3, W7, W11
  - V53I: V4, V8, V12
- **Imaging**: Each biological replicate seeded in one chamber slide with ~10 images of different views
- **Input files**: CZI format (Zeiss)
- **Channels**:
  - DAPI = nucleus
  - NIR = WGA-770
  - AF488 = phalloidin
  - AF647 = GM130 (Golgi)
  - AF568 = CCM2-V5
2. **Save plots** (ROC and top-features barplot) with filenames reflecting **condition / strategy / matching / level**.
3. Feature category naming simplified:
   - `RadialDistribution_*` → **RadialDistribution**
   - `Localization_*` (Location, MassDisplacement, CenterMassIntensity, EdgeEnrichment, remaining non-abundance Intensity) → **Localization**


In [ ]:
# ---- Imports ----
from pathlib import Path
import gc
import re

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


In [ ]:
from typing import Optional, List, Dict, Tuple


## Paths, outputs, folds

In [ ]:
# DATA_DIR: Path to CellProfiler merged output CSVs
# Update this path to point to your local CellProfiler output directory
DATA_DIR = Path("data/")

ANALYSIS_OUTDIR = Path("results/")
ANALYSIS_OUTDIR.mkdir(parents=True, exist_ok=True)

FIG_OUTDIR = ANALYSIS_OUTDIR / "figures_10f_v7"
FIG_OUTDIR.mkdir(parents=True, exist_ok=True)

print("Analysis output dir:", ANALYSIS_OUTDIR)
print("Figure output dir:", FIG_OUTDIR)

CONDITION_KEYWORDS = {"STATIC": "STATIC", "2HR": "2HR", "24HR": "24HR"}

WT_BIOREPS = ["W3", "W7", "W11"]
V_BIOREPS  = ["V4", "V8", "V12"]
CV_FOLDS = [(w, v) for w in WT_BIOREPS for v in V_BIOREPS]
print("CV folds:", CV_FOLDS)

DEFAULT_HEATMAP_FEATURES = [
    "Cells_Granularity_1_CCM2", "Cells_Granularity_2_CCM2", "Cells_Granularity_3_CCM2",
    "Cells_Granularity_4_CCM2", "Cells_Granularity_5_CCM2", "Cells_Granularity_6_CCM2",
    "Cells_Location_MaxIntensity_Y_CCM2",
    "Cells_RadialDistribution_FracAtD_CCM2_10of10", "Cells_RadialDistribution_FracAtD_CCM2_9of10",
    "Cells_RadialDistribution_FracAtD_CCM2_2of10", "Cells_RadialDistribution_FracAtD_CCM2_1of10",
    "Cells_RadialDistribution_FracAtD_CCM2_4of4", "Cells_RadialDistribution_FracAtD_CCM2_1of4",
]

def safe_stem(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s.strip())
    return re.sub(r"_+", "_", s)

def save_fig(stem: str, dpi=300):
    path = FIG_OUTDIR / f"{safe_stem(stem)}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print("Saved figure:", path)
    plt.show()
    plt.close()


## Locate compartment CSVs

In [ ]:
def find_compartment_csv(data_dir: Path, compart: str) -> Path:
    hits = sorted(data_dir.rglob(f"*{compart}*.csv"))
    if not hits:
        raise FileNotFoundError(f"No CSV found for compartment '{compart}' under {data_dir}")
    return hits[0]

CELLS_CSV = find_compartment_csv(DATA_DIR, "Cells")
NUCLEI_CSV = find_compartment_csv(DATA_DIR, "Nuclei")
CYTOPLASM_CSV = find_compartment_csv(DATA_DIR, "Cytoplasm")

print("Cells:", CELLS_CSV)
print("Nuclei:", NUCLEI_CSV)
print("Cytoplasm:", CYTOPLASM_CSV)


## Loader utilities (RAM-efficient; selective columns)

In [ ]:
def _norm_upper(col: pl.Expr) -> pl.Expr:
    return col.cast(pl.Utf8).str.to_uppercase()

def load_compartment_ccm2_condition(csv_path: Path, compart_prefix: str, condition_keyword: str) -> pl.DataFrame:
    header = pl.read_csv(csv_path, n_rows=0)
    cols = header.columns

    key_cols = [c for c in ["ImageNumber", "ObjectNumber"] if c in cols]
    meta_cols = [c for c in cols if c.startswith("Metadata_")]

    ccm2_cols = []
    for c in cols:
        if "_CCM2" not in c:  # includes RadialDistribution_*_CCM2_10of10
            continue
        if c.startswith("Metadata_") or c in key_cols:
            continue
        if ("Parent" in c) or ("Children" in c) or ("Number_Object" in c):
            continue
        if ("FileName" in c) or ("PathName" in c) or ("Name" in c):
            continue
        ccm2_cols.append(c)

    use_cols = [c for c in (key_cols + meta_cols + ccm2_cols) if c in cols]
    df = pl.read_csv(csv_path, columns=use_cols)

    # Prefix non-metadata features
    rename = {}
    for c in df.columns:
        if c.startswith("Metadata_") or c in key_cols:
            continue
        if not c.startswith(f"{compart_prefix}_"):
            rename[c] = f"{compart_prefix}_{c}"
    if rename:
        df = df.rename(rename)

    # Condition filter
    if "Metadata_Condition" not in df.columns:
        raise ValueError("Metadata_Condition missing.")
    df = df.filter(_norm_upper(pl.col("Metadata_Condition")).str.contains(condition_keyword.upper()))

    # BioRep/Genotype
    if "Metadata_Genotype" not in df.columns:
        raise ValueError("Metadata_Genotype missing.")
    df = df.with_columns([
        pl.col("Metadata_Genotype").cast(pl.Utf8).alias("BioRep"),
        pl.when(pl.col("Metadata_Genotype").cast(pl.Utf8).str.starts_with("W")).then(pl.lit("WT"))
         .when(pl.col("Metadata_Genotype").cast(pl.Utf8).str.starts_with("V")).then(pl.lit("V53I"))
         .otherwise(pl.lit("UNKNOWN")).alias("Genotype"),
    ])

    if "Metadata_ImageNumber" not in df.columns and "ImageNumber" in df.columns:
        df = df.with_columns(pl.col("ImageNumber").alias("Metadata_ImageNumber"))

    return df

def load_cells_only(condition_keyword: str) -> pl.DataFrame:
    return load_compartment_ccm2_condition(CELLS_CSV, "Cells", condition_keyword)

def load_all_compartments_merged(condition_keyword: str) -> pl.DataFrame:
    nuc = load_compartment_ccm2_condition(NUCLEI_CSV, "Nuclei", condition_keyword)
    cyto = load_compartment_ccm2_condition(CYTOPLASM_CSV, "Cytoplasm", condition_keyword)
    cells = load_compartment_ccm2_condition(CELLS_CSV, "Cells", condition_keyword)

    base = cells
    for other in [cyto, nuc]:
        key_cols = [c for c in ["ImageNumber", "ObjectNumber"] if c in base.columns and c in other.columns]
        if len(key_cols) < 2:
            raise ValueError("Need ImageNumber and ObjectNumber in all tables to merge.")
        add_cols = key_cols + [c for c in other.columns if c not in base.columns]
        base = base.join(other.select(add_cols), on=key_cols, how="inner")
    return base

def pick_abundance_col(cols: list[str]) -> str:
    candidates = [c for c in cols if "Intensity_IntegratedIntensity_CCM2" in c]
    for c in candidates:
        if c.startswith("Cells_"):
            return c
    if candidates:
        return candidates[0]
    raise ValueError("No column containing 'Intensity_IntegratedIntensity_CCM2' found.")


## Feature selection + simplified categories

In [ ]:
from typing import Optional, Tuple

ABUNDANCE_INTENSITY_TOKENS = [
    "IntegratedIntensity", "MeanIntensity", "MedianIntensity", "MaxIntensity", "MinIntensity",
    "StdIntensity", "MADIntensity", "UpperQuartileIntensity", "LowerQuartileIntensity",
    "TotalIntensity", "SumIntensity",
]

def is_abundance_intensity_feature(feat: str) -> bool:
    """
    Heuristic: remove intensity-summarizing 'abundance' features while keeping
    localization-like intensity readouts (edge, center-of-mass intensity, mass displacement).
    """
    # Keep localization-like intensity readouts
    if "IntensityEdge" in feat:
        return False
    if "MassDisplacement" in feat or "CenterMassIntensity" in feat:
        return False

    # Only consider true intensity summaries
    if "_Intensity_" not in feat:
        return False

    return any(tok in feat for tok in ABUNDANCE_INTENSITY_TOKENS)

def select_ccm2_features(
    df: pd.DataFrame,
    abundance_col: str,
    prefix_allow: Optional[Tuple[str, ...]] = None,
    remove_abundance_intensity: bool = False,
) -> list:
    """
    Select numeric CCM2 features.
    - Must contain '_CCM2' anywhere (captures RadialDistribution_*_CCM2_10of10)
    - Exclude abundance_col (used only for matching)
    - Optionally restrict by prefix (e.g., ('Cells_',))
    - Optionally remove abundance/intensity-summary features (heuristic)
    """
    meta_cols = [c for c in df.columns if c.startswith("Metadata_")] + [
        "ImageNumber", "ObjectNumber", "BioRep", "Genotype"
    ]
    meta_cols = [c for c in meta_cols if c in df.columns]

    feats = []
    for c in df.columns:
        if c in meta_cols:
            continue
        if "_CCM2" not in c:  # IMPORTANT: includes RadialDistribution_*_CCM2_10of10
            continue
        if c == abundance_col:
            continue
        if prefix_allow is not None and not c.startswith(prefix_allow):
            continue
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        if remove_abundance_intensity and is_abundance_intensity_feature(c):
            continue
        feats.append(c)
    return feats

def parse_compartment(feat: str) -> str:
    for pref in ["Cells_", "Cytoplasm_", "Nuclei_"]:
        if feat.startswith(pref):
            return pref[:-1]
    return "Unknown"

def feature_category(feat: str) -> str:
    """
    Category naming per your v7 update:
    - RadialDistribution_* (or MeasureObjectIntensityDistribution_*) -> 'RadialDistribution'
    - Location / MassDisplacement / CenterMassIntensity / IntensityEdge / remaining non-abundance Intensity_ -> 'Localization'
    - Granularity / Texture / Correlation preserved
    """
    base = feat
    for pref in ["Cells_", "Cytoplasm_", "Nuclei_"]:
        if base.startswith(pref):
            base = base[len(pref):]
            break

    # 1) Radial distribution (explicit)
    if base.startswith("RadialDistribution_") or base.startswith("MeasureObjectIntensityDistribution_"):
        return "RadialDistribution"

    # 2) Localization (combined)
    if (
        base.startswith("Location_")
        or base.startswith("IntensityEdge")
        or ("MassDisplacement" in base)
        or ("CenterMassIntensity" in base)
        or base.startswith("Intensity_")
    ):
        return "Localization"

    # 3) Other named families
    if base.startswith("Granularity_"):
        return "Granularity"
    if base.startswith("Texture_"):
        return "Texture"
    if base.startswith("Correlation_"):
        return "Correlation"

    # 4) fallback
    return base.split("_")[0] if "_" in base else base

def feature_catalog_table(feature_cols: list) -> pd.DataFrame:
    out = pd.DataFrame({
        "feature": feature_cols,
        "compartment": [parse_compartment(f) for f in feature_cols],
        "category": [feature_category(f) for f in feature_cols],
    })
    return out.sort_values(["category", "compartment", "feature"]).reset_index(drop=True)

def plot_feature_category_counts(feature_cols: list, title: str, save_stem: Optional[str] = None):
    fam = pd.Series([feature_category(f) for f in feature_cols])
    counts = fam.value_counts().sort_values(ascending=True)

    plt.figure(figsize=(8, max(3, 0.35 * len(counts))))
    plt.barh(counts.index.tolist(), counts.values.tolist())
    plt.xlabel("Number of features retained")
    plt.title(title)
    plt.tight_layout()

    if save_stem is not None:
        save_fig(save_stem)
    else:
        plt.show()
        plt.close()

## QC, preprocessing, CV, ROC plotting (with saving)

In [ ]:
# -----------------------------
# QC, preprocessing, CV, ROC plotting (Py<3.10 safe)
# -----------------------------
from typing import Optional, Dict, Tuple, List

def detect_noncontinuous_features(
    train_df: pd.DataFrame,
    feature_cols: List[str],
    max_unique_for_categorical: int = 10,
    low_variance_thresh: float = 1e-12
) -> Tuple[List[str], pd.DataFrame]:
    dropped = []
    for f in feature_cols:
        s = train_df[f]

        if s.isna().all():
            dropped.append((f, "all_null"))
            continue

        nonnull = s.dropna()
        if len(nonnull) == 0:
            dropped.append((f, "all_null"))
            continue

        if (nonnull == 0).all():
            dropped.append((f, "all_zero"))
            continue

        nunique = nonnull.nunique(dropna=True)
        if nunique <= 1:
            dropped.append((f, "constant"))
            continue

        uniq = sorted(pd.unique(nonnull))
        if len(uniq) == 2 and uniq[0] == 0 and uniq[1] == 1:
            dropped.append((f, "binary_0_1"))
            continue

        if nunique <= max_unique_for_categorical:
            dropped.append((f, f"low_unique<= {max_unique_for_categorical}"))
            continue

        if np.nanvar(nonnull.to_numpy(dtype=float)) <= low_variance_thresh:
            dropped.append((f, "extremely_low_variance"))
            continue

    dropped_df = pd.DataFrame(dropped, columns=["feature", "reason"])
    dropped_set = set(dropped_df["feature"].tolist())
    keep = [f for f in feature_cols if f not in dropped_set]
    return keep, dropped_df

def winsorize_fit(
    train_df: pd.DataFrame,
    feature_cols: List[str],
    lo: float = 0.005,
    hi: float = 0.995
) -> Dict[str, Tuple[float, float]]:
    qs = {}
    for f in feature_cols:
        x = train_df[f].astype(float)
        qs[f] = (np.nanquantile(x, lo), np.nanquantile(x, hi))
    return qs

def winsorize_apply(
    df: pd.DataFrame,
    feature_cols: List[str],
    qs: Dict[str, Tuple[float, float]]
) -> pd.DataFrame:
    out = df.copy()
    for f in feature_cols:
        lo, hi = qs[f]
        out[f] = out[f].astype(float).clip(lo, hi)
    return out

def robust_scale_fit(
    train_df: pd.DataFrame,
    feature_cols: List[str]
) -> Dict[str, Tuple[float, float]]:
    params = {}
    for f in feature_cols:
        x = train_df[f].astype(float)
        med = np.nanmedian(x)
        iqr = np.nanpercentile(x, 75) - np.nanpercentile(x, 25)
        if iqr == 0 or np.isnan(iqr):
            iqr = 1.0
        params[f] = (med, iqr)
    return params

def robust_scale_apply(
    df: pd.DataFrame,
    feature_cols: List[str],
    params: Dict[str, Tuple[float, float]]
) -> pd.DataFrame:
    out = df.copy()
    for f in feature_cols:
        med, iqr = params[f]
        out[f] = (out[f].astype(float) - med) / iqr
    return out

def corr_filter_fit(
    X: pd.DataFrame,
    threshold: float = 0.90
) -> Tuple[List[str], List[str]]:
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    keep = [c for c in X.columns if c not in to_drop]
    return keep, to_drop

def preprocess_fit_transform(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: List[str],
    clip_lo: float = 0.005,
    clip_hi: float = 0.995,
    variance_thresh: float = 0.0,
    corr_thresh: float = 0.90
):
    y_train = (train_df["Genotype"] == "V53I").astype(int)
    y_test  = (test_df["Genotype"] == "V53I").astype(int)

    # winsorize
    qs = winsorize_fit(train_df, feature_cols, lo=clip_lo, hi=clip_hi)
    train_w = winsorize_apply(train_df, feature_cols, qs)
    test_w  = winsorize_apply(test_df, feature_cols, qs)

    # fill NaNs using train medians
    medians = {f: np.nanmedian(train_w[f].astype(float)) for f in feature_cols}
    for f in feature_cols:
        train_w[f] = train_w[f].astype(float).fillna(medians[f])
        test_w[f]  = test_w[f].astype(float).fillna(medians[f])

    # robust scale
    scaler_params = robust_scale_fit(train_w, feature_cols)
    train_s = robust_scale_apply(train_w, feature_cols, scaler_params)
    test_s  = robust_scale_apply(test_w, feature_cols, scaler_params)

    X_train = train_s[feature_cols].copy()
    X_test  = test_s[feature_cols].copy()

    # variance threshold
    vt = VarianceThreshold(threshold=variance_thresh)
    vt.fit(X_train)
    kept_vt = X_train.columns[vt.get_support()].tolist()
    X_train = X_train[kept_vt]
    X_test  = X_test[kept_vt]

    # correlation filter
    kept_corr, dropped_corr = corr_filter_fit(X_train, threshold=corr_thresh)
    X_train = X_train[kept_corr]
    X_test  = X_test[kept_corr]

    artifact = {"kept_features": kept_corr, "dropped_corr": dropped_corr, "kept_vt": kept_vt}
    return X_train, y_train, X_test, y_test, artifact

def train_xgb_classifier(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    random_state: int = 0
):
    clf = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=4,
        random_state=random_state
    )
    clf.fit(X_train, y_train)
    return clf

def plot_mean_roc(
    fprs,
    tprs,
    aucs,
    title: str,
    save_stem: Optional[str] = None
):
    mean_fpr = np.linspace(0, 1, 200)
    interp_tprs = []

    for fpr, tpr in zip(fprs, tprs):
        interp = np.interp(mean_fpr, fpr, tpr)
        interp[0] = 0.0
        interp_tprs.append(interp)

    if len(interp_tprs) == 0:
        mean_tpr = np.zeros_like(mean_fpr)
        std_tpr = np.zeros_like(mean_fpr)
    else:
        mean_tpr = np.mean(interp_tprs, axis=0)
        std_tpr = np.std(interp_tprs, axis=0)
        mean_tpr[-1] = 1.0

    mean_auc = float(np.mean(aucs)) if len(aucs) else float("nan")
    std_auc  = float(np.std(aucs)) if len(aucs) else float("nan")

    plt.figure()
    for fpr, tpr in zip(fprs, tprs):
        plt.plot(fpr, tpr, alpha=0.25)

    plt.plot(mean_fpr, mean_tpr, linewidth=2)
    plt.fill_between(
        mean_fpr,
        np.maximum(mean_tpr - std_tpr, 0),
        np.minimum(mean_tpr + std_tpr, 1),
        alpha=0.2
    )
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title}\nMean AUC = {mean_auc:.3f} ± {std_auc:.3f} (n={len(aucs)} folds)")
    plt.tight_layout()

    if save_stem is not None:
        save_fig(save_stem)
    else:
        plt.show()
        plt.close()

    return mean_auc, std_auc


## Matching + aggregation + CV
CV = Cross-validation

In [ ]:
def iqr_overlap_match_global(df: pd.DataFrame, abundance_col: str, qlo=25.0, qhi=75.0):
    wt = df.loc[df["Genotype"]=="WT", abundance_col].astype(float)
    vv = df.loc[df["Genotype"]=="V53I", abundance_col].astype(float)
    q1_wt, q3_wt = np.nanpercentile(wt, [qlo, qhi])
    q1_v,  q3_v  = np.nanpercentile(vv, [qlo, qhi])
    lower = max(q1_wt, q1_v)
    upper = min(q3_wt, q3_v)
    info = {"overlap": (lower, upper), "wt_q": (q1_wt, q3_wt), "v_q": (q1_v, q3_v)}
    matched = df[df[abundance_col].astype(float).between(lower, upper)].copy()
    return matched, info

def make_train_test_split(df: pd.DataFrame, test_w: str, test_v: str):
    test_bioreps = [test_w, test_v]
    train_bioreps = [b for b in df["BioRep"].unique().tolist() if b not in test_bioreps]
    train_df = df[df["BioRep"].isin(train_bioreps)].copy()
    test_df  = df[df["BioRep"].isin(test_bioreps)].copy()
    return train_df, test_df

def fold_specific_iqr_filter(train_df: pd.DataFrame, test_df: pd.DataFrame, abundance_col: str, qlo=25.0, qhi=75.0):
    wt = train_df.loc[train_df["Genotype"]=="WT", abundance_col].astype(float)
    vv = train_df.loc[train_df["Genotype"]=="V53I", abundance_col].astype(float)
    q1_wt, q3_wt = np.nanpercentile(wt, [qlo, qhi])
    q1_v,  q3_v  = np.nanpercentile(vv, [qlo, qhi])
    lower = max(q1_wt, q1_v)
    upper = min(q3_wt, q3_v)
    train_f = train_df[train_df[abundance_col].astype(float).between(lower, upper)].copy()
    test_f  = test_df[test_df[abundance_col].astype(float).between(lower, upper)].copy()
    return train_f, test_f, {"overlap": (lower, upper)}

def aggregate_to_image(df: pd.DataFrame, feature_cols: list[str]):
    group_cols = ["BioRep","Genotype","Metadata_ImageNumber"]
    return df[group_cols + feature_cols].groupby(group_cols)[feature_cols].median().reset_index()

def run_biorep_pair_cv(df: pd.DataFrame, feature_cols: list[str], abundance_col: str, level="cell", use_fold_matching=False):
    aucs, fprs, tprs = [], [], []
    for w_test, v_test in CV_FOLDS:
        train_df, test_df = make_train_test_split(df, w_test, v_test)
        if use_fold_matching:
            train_df, test_df, _ = fold_specific_iqr_filter(train_df, test_df, abundance_col)

        if level == "image":
            train_use = aggregate_to_image(train_df, feature_cols)
            test_use  = aggregate_to_image(test_df, feature_cols)
        else:
            train_use, test_use = train_df, test_df

        keep, _ = detect_noncontinuous_features(train_use, feature_cols)
        if len(keep) == 0:
            continue

        Xtr, ytr, Xte, yte, _ = preprocess_fit_transform(train_use, test_use, keep)
        model = train_xgb_classifier(Xtr, ytr, random_state=0)
        proba = model.predict_proba(Xte)[:,1]
        aucs.append(roc_auc_score(yte, proba))
        fpr, tpr, _ = roc_curve(yte, proba)
        fprs.append(fpr); tprs.append(tpr)

    return aucs, fprs, tprs


## Stats + top-features plotting (with saving)

In [ ]:
# -----------------------------
# Stats + top-features plotting (with saving) — Py<3.10 safe
# -----------------------------
from typing import Optional, List, Dict

def differential_stats(
    df: pd.DataFrame,
    feature_cols: List[str],
    g1: str = "WT",
    g2: str = "V53I"
) -> pd.DataFrame:
    eps = 1e-9
    a = df[df["Genotype"] == g1]
    b = df[df["Genotype"] == g2]

    rows = []
    for f in feature_cols:
        x = a[f].astype(float).to_numpy()
        y = b[f].astype(float).to_numpy()
        x = x[~np.isnan(x)]
        y = y[~np.isnan(y)]

        if len(x) < 3 or len(y) < 3:
            p = np.nan
        else:
            try:
                p = mannwhitneyu(x, y, alternative="two-sided").pvalue
            except Exception:
                p = np.nan

        mean_x = np.nanmean(x) if len(x) else np.nan
        mean_y = np.nanmean(y) if len(y) else np.nan
        med_x  = np.nanmedian(x) if len(x) else np.nan
        med_y  = np.nanmedian(y) if len(y) else np.nan

        sd_x = np.nanstd(x, ddof=1) if len(x) > 1 else np.nan
        sd_y = np.nanstd(y, ddof=1) if len(y) > 1 else np.nan

        se_x = (sd_x / np.sqrt(len(x))) if len(x) > 1 and not np.isnan(sd_x) else np.nan
        se_y = (sd_y / np.sqrt(len(y))) if len(y) > 1 and not np.isnan(sd_y) else np.nan

        mx_x = np.nanmax(x) if len(x) else np.nan
        mx_y = np.nanmax(y) if len(y) else np.nan
        mn_x = np.nanmin(x) if len(x) else np.nan
        mn_y = np.nanmin(y) if len(y) else np.nan

        logfc = (np.log2(mean_y + eps) - np.log2(mean_x + eps)) if (not np.isnan(mean_x) and not np.isnan(mean_y)) else np.nan

        rows.append({
            "feature": f,
            "p_value": p,
            "p_adj_bh": np.nan,
            "logFC_log2_mean": logfc,
            "WT_mean": mean_x, "V53I_mean": mean_y,
            "WT_median": med_x, "V53I_median": med_y,
            "WT_sd": sd_x, "V53I_sd": sd_y,
            "WT_se": se_x, "V53I_se": se_y,
            "WT_max": mx_x, "V53I_max": mx_y,
            "WT_min": mn_x, "V53I_min": mn_y,
            "WT_n": int(len(x)), "V53I_n": int(len(y)),
        })

    res = pd.DataFrame(rows)
    pvals = res["p_value"].to_numpy()
    mask = ~np.isnan(pvals)
    if mask.sum() > 0:
        res.loc[mask, "p_adj_bh"] = multipletests(pvals[mask], method="fdr_bh")[1]
    return res.sort_values(["p_adj_bh", "p_value"], na_position="last")

def fit_final_model_and_top_features(
    df: pd.DataFrame,
    feature_cols: List[str],
    title_prefix: str,
    level: str = "cell",
    top_n: int = 20,
    save_stem: Optional[str] = None
) -> Dict:
    """
    Fits a single model on the provided df (cell or image aggregated),
    and saves a top-features barplot if save_stem is provided.

    Returns a dict with model, importances, top_features, dropped_qc.
    """
    # aggregate if needed
    if level == "image":
        df_use = aggregate_to_image(df, feature_cols)
    else:
        df_use = df.copy()

    # value-based QC
    keep, dropped = detect_noncontinuous_features(df_use, feature_cols)

    # save kept feature lists (optional, but handy)
    kept_tbl = feature_catalog_table(keep)
    kept_tbl.to_csv(
        ANALYSIS_OUTDIR / f"{safe_stem(title_prefix)}_{level}_keptFeatures_valueQC.csv",
        index=False
    )

    # preprocess (fit on df_use itself, for top-features summary)
    Xtr, ytr, _, _, artifact = preprocess_fit_transform(df_use, df_use, keep)

    final_kept = artifact["kept_features"]
    final_tbl = feature_catalog_table(final_kept)
    final_tbl.to_csv(
        ANALYSIS_OUTDIR / f"{safe_stem(title_prefix)}_{level}_keptFeatures_afterPreprocess.csv",
        index=False
    )

    # fit model
    model = train_xgb_classifier(Xtr, ytr, random_state=0)

    # feature importance
    imp = pd.Series(model.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
    top = imp.head(top_n)

    # plot
    plt.figure(figsize=(6, 6))
    top.sort_values().plot(kind="barh")
    plt.title(f"{title_prefix} Top {top_n} features ({level})")
    plt.xlabel("XGBoost feature importance")
    plt.tight_layout()

    if save_stem is not None:
        save_fig(save_stem)
    else:
        plt.show()
        plt.close()

    return {
        "model": model,
        "importances": imp,
        "top_features": top.index.tolist(),
        "dropped_qc": dropped
    }


## Custom heatmap helper 

In [ ]:
# -----------------------------
# Custom heatmap helper (Py<3.10 safe)
# -----------------------------
from typing import Optional, List

def plot_custom_heatmap_from_features(
    df: pd.DataFrame,
    feature_list: List[str],
    title: str,
    level: str = "cell",
    preprocess_like_model: bool = True,
    save_stem: Optional[str] = None
):
    """
    Heatmap of genotype means for a chosen feature list.
    - level: 'cell' or 'image' (image = median aggregation per Metadata_ImageNumber)
    - preprocess_like_model:
        True  -> winsorize + robust-scale like your modeling pipeline before visualization
        False -> raw values (still z-scored per feature for plotting)
    - Visualization scaling: z-score PER FEATURE across samples, then mean by genotype.
      This is what you wanted: normalize across features, not converting WT/V53I to 0/1.
    """
    feats_present = [f for f in feature_list if f in df.columns]
    missing = [f for f in feature_list if f not in df.columns]
    if missing:
        print("NOTE: missing features (skipped):")
        for m in missing:
            print("  -", m)
    if len(feats_present) == 0:
        print("No requested features found; cannot plot.")
        return

    # Aggregate to image-level if requested
    if level == "image":
        df_use = aggregate_to_image(df, feats_present)
    else:
        df_use = df.copy()

    # Ensure numeric + fill NaNs
    X = df_use[feats_present].copy()
    for f in feats_present:
        X[f] = pd.to_numeric(X[f], errors="coerce")
        X[f] = X[f].fillna(np.nanmedian(X[f].values))

    # Optionally apply the same style of preprocessing as model
    if preprocess_like_model:
        qs = winsorize_fit(df_use, feats_present, lo=0.005, hi=0.995)
        df_w = winsorize_apply(df_use, feats_present, qs)
        scaler = robust_scale_fit(df_w, feats_present)
        df_s = robust_scale_apply(df_w, feats_present, scaler)
        X = df_s[feats_present].astype(float)

    # Z-score per feature across samples (so features become comparable for heatmap)
    mu = X.mean(axis=0).values.reshape(1, -1)
    sd = X.std(axis=0, ddof=0).values.reshape(1, -1)
    sd[sd == 0] = 1.0
    Xz = (X.values - mu) / sd

    # Mean by genotype (rows WT/V53I)
    geno_means = (
        pd.DataFrame(Xz, columns=feats_present)
        .assign(Genotype=df_use["Genotype"].values)
        .groupby("Genotype")[feats_present]
        .mean()
    )

    # Plot
    plt.figure(figsize=(min(18, 0.55 * len(feats_present) + 4), 3.4))
    im = plt.imshow(geno_means.values, aspect="auto", cmap="bwr")  # blue-low red-high
    plt.yticks(range(len(geno_means.index)), geno_means.index.tolist())
    plt.xticks(range(len(feats_present)), feats_present, rotation=90)
    plt.title(title)

    cbar = plt.colorbar(im)
    cbar.set_label("Z-score (per feature across samples)")

    plt.tight_layout()

    if save_stem is not None:
        save_fig(save_stem)
    else:
        plt.show()
        plt.close()


## Main analysis (one strategy at a time; saves ROC/top-features)

In [ ]:
def analyze_condition_one_strategy(condition_label: str, keyword: str, strategy_name: str, loader_fn):
    pl_df = loader_fn(keyword)
    pl_df = pl_df.filter(pl.col("Genotype").is_in(["WT", "V53I"]))
    df = pl_df.to_pandas()
    del pl_df
    gc.collect()

    abundance_col = pick_abundance_col(df.columns.tolist())

    if strategy_name == "S1_CellsDistributionOnly":
        feats = select_ccm2_features(df, abundance_col, prefix_allow=("Cells_",), remove_abundance_intensity=True)
    elif strategy_name == "S2_AllCompartmentsDistributionOnly":
        feats = select_ccm2_features(df, abundance_col, prefix_allow=("Cells_","Cytoplasm_","Nuclei_"), remove_abundance_intensity=True)
    else:
        raise ValueError("Unknown strategy")

    title_base = f"{condition_label}_{strategy_name}"

    # Candidate category plot + catalog
    feature_catalog_table(feats).to_csv(ANALYSIS_OUTDIR / f"{title_base}_candidateFeatures_catalog.csv", index=False)
    plot_feature_category_counts(feats, title=f"{condition_label} {strategy_name}: candidate categories",
                                save_stem=f"{title_base}_candidate_featureCategories")

    # --- No matching ---
    w0, v0 = CV_FOLDS[0]
    train0, _ = make_train_test_split(df, w0, v0)
    keep0, dropped0 = detect_noncontinuous_features(train0, feats)
    dropped0.to_csv(ANALYSIS_OUTDIR / f"{title_base}_noMatching_exampleFold_droppedQC.csv", index=False)

    plot_feature_category_counts(keep0, title=f"{condition_label} {strategy_name}: after value-QC (example fold)",
                                save_stem=f"{title_base}_noMatching_afterQC_featureCategories")

    # Variance distribution (example fold)
    var = np.nanvar(train0[keep0].to_numpy(dtype=float), axis=0)
    var = var[var > 0]
    plt.figure()
    plt.hist(np.log10(var), bins=50)
    plt.xlabel("log10(variance)")
    plt.ylabel("Number of features")
    plt.title(f"{condition_label} {strategy_name} (no matching): variance distribution (example fold)")
    plt.tight_layout()
    save_fig(f"{title_base}_noMatching_varianceDistribution_exampleFold")

    # CV ROC (saved)
    aucs_c, fprs_c, tprs_c = run_biorep_pair_cv(df, feats, abundance_col, level="cell", use_fold_matching=False)
    plot_mean_roc(fprs_c, tprs_c, aucs_c,
                  title=f"{condition_label} {strategy_name} — Cell-level (no matching)",
                  save_stem=f"{title_base}_noMatching_cellLevel_ROC")

    aucs_i, fprs_i, tprs_i = run_biorep_pair_cv(df, feats, abundance_col, level="image", use_fold_matching=False)
    plot_mean_roc(fprs_i, tprs_i, aucs_i,
                  title=f"{condition_label} {strategy_name} — Image-level (no matching)",
                  save_stem=f"{title_base}_noMatching_imageLevel_ROC")

    # Top features plots (saved)
    fit_final_model_and_top_features(
        df, feats, title_prefix=f"{condition_label} {strategy_name} noMatching", level="cell", top_n=20,
        save_stem=f"{title_base}_noMatching_cellLevel_topFeatures"
    )
    fit_final_model_and_top_features(
        df, feats, title_prefix=f"{condition_label} {strategy_name} noMatching", level="image", top_n=20,
        save_stem=f"{title_base}_noMatching_imageLevel_topFeatures"
    )

    # --- Fold-specific matching (Option A) ---
    df_gm, info = iqr_overlap_match_global(df, abundance_col)

    aucs_c_m, fprs_c_m, tprs_c_m = run_biorep_pair_cv(df, feats, abundance_col, level="cell", use_fold_matching=True)
    plot_mean_roc(fprs_c_m, tprs_c_m, aucs_c_m,
                  title=f"{condition_label} {strategy_name} — Cell-level (fold IQR match)",
                  save_stem=f"{title_base}_foldMatch_cellLevel_ROC")

    aucs_i_m, fprs_i_m, tprs_i_m = run_biorep_pair_cv(df, feats, abundance_col, level="image", use_fold_matching=True)
    plot_mean_roc(fprs_i_m, tprs_i_m, aucs_i_m,
                  title=f"{condition_label} {strategy_name} — Image-level (fold IQR match)",
                  save_stem=f"{title_base}_foldMatch_imageLevel_ROC")

    # Top features plots on global matched dataset (descriptive; saved)
    fit_final_model_and_top_features(
        df_gm, feats, title_prefix=f"{condition_label} {strategy_name} globalMatched", level="cell", top_n=20,
        save_stem=f"{title_base}_globalMatched_cellLevel_topFeatures"
    )
    fit_final_model_and_top_features(
        df_gm, feats, title_prefix=f"{condition_label} {strategy_name} globalMatched", level="image", top_n=20,
        save_stem=f"{title_base}_globalMatched_imageLevel_topFeatures"
    )

    return {"df": df, "df_global_matched": df_gm, "abundance_col": abundance_col, "candidate_features": feats}


## STATIC — S1 (Cells only) run

In [ ]:
res_STATIC_S1 = analyze_condition_one_strategy(
    condition_label="STATIC",
    keyword=CONDITION_KEYWORDS["STATIC"],
    strategy_name="S1_CellsDistributionOnly",
    loader_fn=load_cells_only
)


## STATIC — S1 custom heatmap (edit HEATMAP_FEATURES)

In [ ]:
HEATMAP_FEATURES = list(DEFAULT_HEATMAP_FEATURES)  # <-- edit freely
df_for_heatmap = res_STATIC_S1["df_global_matched"]
plot_custom_heatmap_from_features(
    df_for_heatmap,
    HEATMAP_FEATURES,
    title=f"STATIC S1 (Cells) — global matched — custom heatmap",
    level="cell",
    preprocess_like_model=True,
    save_stem=f"STATIC_S1_globalMatched_customHeatmap_cell"
)


## STATIC — S2 (All compartments) run

In [ ]:
res_STATIC_S2 = analyze_condition_one_strategy(
    condition_label="STATIC",
    keyword=CONDITION_KEYWORDS["STATIC"],
    strategy_name="S2_AllCompartmentsDistributionOnly",
    loader_fn=load_all_compartments_merged
)


## STATIC — S2 custom heatmap (edit HEATMAP_FEATURES)

In [ ]:
HEATMAP_FEATURES = list(DEFAULT_HEATMAP_FEATURES)  # <-- edit freely
df_for_heatmap = res_STATIC_S2["df_global_matched"]
plot_custom_heatmap_from_features(
    df_for_heatmap,
    HEATMAP_FEATURES,
    title=f"STATIC S2 (All compartments) — global matched — custom heatmap",
    level="cell",
    preprocess_like_model=True,
    save_stem=f"STATIC_S2_globalMatched_customHeatmap_cell"
)


## Notes
- ROC and top-features figures are saved into: `analysis_results/figures_10f_v7/`
- Category simplification:
  - **RadialDistribution** stays separate
  - **Localization** combines Location + MassDisplacement + CenterMassIntensity + EdgeEnrichment + remaining non-abundance Intensity
